<a href="https://colab.research.google.com/github/Syed-Irtaza/CrewLogix_GenAI_Assignment/blob/main/SemanticRAG_Chroma_FastAPI_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Installing libraries**

In [ ]:
!pip install -q langchain langchain-community langchain-openai langchain-experimental langchain-chroma chromadb
!pip install -q pypdf python-docx unstructured fastapi uvicorn pyngrok nest_asyncio

**Adding OPENAI API key**

In [4]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPEN_API_KEY')

**Loading the files (knowldege base) for the RAG**

In [5]:
from google.colab import files
uploaded = files.upload()

Saving new-approaches-and-procedures-for-cancer-treatment.pdf to new-approaches-and-procedures-for-cancer-treatment.pdf
Saving M.Sc. Applied Psychology.docx to M.Sc. Applied Psychology.docx


**Importing the Libraries**

In [9]:
import os
import glob
import shutil
import nest_asyncio
from typing import List

from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

**Configurations: like DB, Model name, Embedding model**

In [10]:
DATA_PATH = "/content"
DB_PATH = "/content/chroma_db"
COLLECTION_NAME = "my_rag_collection"

EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"

**Loading files with LangChain loaders**

In [11]:
def load_single_file(file_path: str) -> List[Document]:
    try:
        if file_path.endswith(".pdf"):
            loader = PyPDFLoader(file_path)
        elif file_path.endswith(".txt"):
            loader = TextLoader(file_path, encoding="utf-8")
        elif file_path.endswith(".docx"):
            loader = Docx2txtLoader(file_path)
        else:
            print(f"Unsupported file skipped: {file_path}")
            return []

        return loader.load()

    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return []


def load_all_documents(folder_path: str) -> List[Document]:
    documents = []

    allowed_extensions = ["*.pdf", "*.txt", "*.docx"]

    for ext in allowed_extensions:
        for file_path in glob.glob(os.path.join(folder_path, ext)):
            documents.extend(load_single_file(file_path))

    if not documents:
        raise ValueError("No valid documents found. Upload PDF, TXT, or DOCX files.")

    return documents

**Semantic chunking**

In [12]:
def semantic_chunk_documents(documents: List[Document]) -> List[Document]:
    try:
        embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

        splitter = SemanticChunker(
            embeddings,
            breakpoint_threshold_type="percentile"
        )

        chunks = splitter.split_documents(documents)

        if not chunks:
            raise ValueError("Chunking failed. No chunks were created.")

        return chunks

    except Exception as e:
        raise RuntimeError(f"Semantic chunking error: {e}")

**Storing chunks in persistent Chroma DB**

In [13]:
def create_chroma_db(chunks: List[Document]):
    try:
        embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

        if os.path.exists(DB_PATH):
            shutil.rmtree(DB_PATH)

        vector_db = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            collection_name=COLLECTION_NAME,
            persist_directory=DB_PATH
        )

        return vector_db

    except Exception as e:
        raise RuntimeError(f"Chroma DB creation error: {e}")

**Loading existing Chroma DB**

In [14]:
def load_chroma_db():
    try:
        embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

        vector_db = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embeddings,
            persist_directory=DB_PATH
        )

        return vector_db

    except Exception as e:
        raise RuntimeError(f"Chroma DB loading error: {e}")

**Build RAG answer function**

In [29]:
def generate_rag_answer(query: str, top_k: int = 3) -> dict:
    try:
        if not query.strip():
            raise ValueError("Query cannot be empty.")

        vector_db = load_chroma_db()

        retriever = vector_db.as_retriever(
            search_kwargs={"k": top_k}
        )

        relevant_docs = retriever.invoke(query)

        if not relevant_docs:
            return {
                "answer": "I could not find relevant information in the provided documents.",
                "sources": []
            }

        context = "\n\n".join([doc.page_content for doc in relevant_docs])

        prompt = ChatPromptTemplate.from_template("""
You are a helpful RAG assistant.

Answer the user question using only the given context.
If the answer is not present in the context, say:
"I don't know based on the provided documents."

Context:
{context}

Question:
{question}

Answer:
""")

        llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

        chain = prompt | llm

        response = chain.invoke({
            "context": context,
            "question": query
        })

        sources = [
            {
                "source": doc.metadata.get("source", "Unknown"),
                "page": doc.metadata.get("page", "N/A")
            }
            for doc in relevant_docs
        ]

        return {
            "answer": response.content,
            "sources": sources
        }

    except Exception as e:
        raise RuntimeError(f"RAG answer generation error: {e}")

**Runing indexing pipeline**

In [30]:
def build_knowledge_base():
    try:
        documents = load_all_documents(DATA_PATH)
        print(f"Loaded documents: {len(documents)}")

        chunks = semantic_chunk_documents(documents)
        print(f"Created chunks: {len(chunks)}")

        create_chroma_db(chunks)
        print("Chroma DB created successfully.")

    except Exception as e:
        print(f"Pipeline error: {e}")


build_knowledge_base()

Error loading /content/M.Sc. Applied Psychology.docx: No module named 'docx2txt'
Loaded documents: 10
Created chunks: 41
Pipeline error: Chroma DB creation error: Query error: Database error: error returned from database: (code: 1032) attempt to write a readonly database


**Testing RAG Locally in colab**

In [31]:
result = generate_rag_answer("What is this pdf about?", top_k=3)
print(result["answer"])
print(result["sources"])

This PDF is about a review that presents an update on recent advances and breakthroughs in cancer therapies.
[{'source': '/content/new-approaches-and-procedures-for-cancer-treatment.pdf', 'page': 4}, {'source': '/content/new-approaches-and-procedures-for-cancer-treatment.pdf', 'page': 6}, {'source': '/content/new-approaches-and-procedures-for-cancer-treatment.pdf', 'page': 0}]


**Creating FastAPI app**

In [23]:
app = FastAPI(title="Semantic RAG API with Chroma DB")


class QueryRequest(BaseModel):
    query: str
    top_k: int = 3


@app.get("/")
def home():
    return {"message": "Semantic RAG API is running."}


@app.post("/ask")
def ask_question(request: QueryRequest):
    try:
        result = generate_rag_answer(
            query=request.query,
            top_k=request.top_k
        )
        return result

    except ValueError as ve:
        raise HTTPException(status_code=400, detail=str(ve))

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

**Runing FastAPI in Colab**

In [27]:
import uvicorn
from pyngrok import ngrok
from google.colab import userdata
import asyncio # Import asyncio

nest_asyncio.apply()

# Set your ngrok authentication token (replace 'YOUR_NGROK_AUTH_TOKEN' with your actual token)
# It's recommended to store this as a Colab secret and retrieve it using userdata.get('NGROK_AUTH_TOKEN')
ngrok_auth_token = userdata.get('NGROK_AUTH_TOKEN') # Assuming you saved your token as 'NGROK_AUTH_TOKEN' in Colab secrets
if ngrok_auth_token:
    ngrok.set_auth_token(ngrok_auth_token)
else:
    print("Ngrok authentication token not found. Please set it as a Colab secret named 'NGROK_AUTH_TOKEN'.")
    # Alternatively, you can directly paste your token here for testing, but it's not recommended for sharing:
    # ngrok.set_auth_token("YOUR_AUTH_TOKEN_HERE")

public_url = ngrok.connect(8000)
print("Public API URL:", public_url)

# Instead of uvicorn.run(), start the server explicitly on the existing event loop
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)

loop = asyncio.get_event_loop()
task = loop.create_task(server.serve())

print("Uvicorn server started in the background. Interrupt kernel to stop.")


Public API URL: NgrokTunnel: "https://flagpole-iodine-diabetes.ngrok-free.dev" -> "http://localhost:8000"
Uvicorn server started in the background. Interrupt kernel to stop.


**Testing ENDPOINT (URL)** = https://flagpole-iodine-diabetes.ngrok-free.dev/docs